# 🎓 Academic Advisor Assistant

**AI Chatbot Development Using LLMs and Gradio**

This notebook builds a chatbot for a **real-time, practical use case**: an academic advisor
assistant for a university's School of Computer & Data Sciences. It uses Python programming language

## Use Case & Scope

**Persona:** Ava, the Academic Advisor Assistant for Cascade State University's School of
Computer & Data Sciences.

**What Ava can help with:**
- Degree requirements for the BS in Computer Science, BS in Data Science, and the AI minor
- Course prerequisites and recommended course sequencing
- Registration policies (credit load limits, add/drop/withdraw deadlines)
- Academic standing rules (GPA requirements, probation, holds)

**Out of scope (by design):** financial aid, medical/personal leave, disciplinary cases,
visa/international status, and anything not in the advising knowledge base below. For these,
Ava redirects the student to the human Advising Office rather than guessing.

**Workflow:**
1. Load a small text-based knowledge base of program/policy facts (`advisor_knowledge.txt`).
2. Fold that knowledge base into a system prompt that defines Ava's persona and boundaries.
3. Validate each user message before it reaches the model (reusable `validate_input` helper).
4. Send `system prompt + history + message` to the OpenAI Chat Completions API.
5. Serve the conversation through a Gradio `ChatInterface` with example prompts.

In [1]:
# Import necessary libraries
from dotenv import load_dotenv 
from openai import OpenAI   
import gradio as gr        

c:\mi\05\sch\CS529\assignments\day-2\AcademicAdvisorChatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

# Create a single global client
client = OpenAI()

## Grounding the Assistant

Ava's answers are grounded in `advisor_knowledge.txt`, a plain-text summary of degree
requirements, prerequisites, and policies.

In [3]:
# Read the advising knowledge base
with open("advisor_knowledge.txt", "r", encoding="utf-8") as f:
    knowledge_base = f.read()

print(knowledge_base)

Institution: Cascade State University — School of Computer & Data Sciences

DEGREE PROGRAMS
- BS in Computer Science (120 credits): Core sequence is CS101 Intro to Programming -> CS201
  Data Structures -> CS301 Algorithms -> CS310 Systems -> CS401 Capstone. Requires one AI/ML
  elective and one Ethics in Computing elective before graduation.
- BS in Data Science (120 credits): Core sequence is DS101 Intro to Data Science -> DS201
  Statistics for Data Science -> DS250 Databases -> DS310 Machine Learning -> DS401 Capstone.
  Requires MATH210 Linear Algebra and MATH220 Probability as prerequisites for DS310.
- Minor in Artificial Intelligence (18 credits): CS201 Data Structures, CS330 Intro to AI,
  CS331 Machine Learning, and two AI electives.

PREREQUISITES
- CS201 Data Structures requires CS101 Intro to Programming (grade C or higher).
- CS301 Algorithms requires CS201 Data Structures.
- DS310 Machine Learning requires DS201 Statistics, MATH210 Linear Algebra, and MATH220
  Probabili

In [4]:
# Design the system prompt: persona + boundaries + the knowledge base as context
advisor_name = "Ava"

system_prompt = (
    f"You are {advisor_name}, the academic advisor assistant for Cascade State University's "
    "School of Computer & Data Sciences. You help students with degree requirements, course "
    "prerequisites, registration policies, and academic standing, using ONLY the advising "
    "knowledge base provided below. Be encouraging, clear, and concise. Ask a clarifying "
    "question when the student's request is ambiguous (e.g. which degree program they are in). "
    "If a question falls outside the knowledge base (financial aid, medical or personal leave, "
    "disciplinary matters, visa/international status, or anything not covered below), do not "
    "guess — politely refer the student to the Advising Office at advising@cascadestate.edu."
)
system_prompt += f"\n\n## Advising Knowledge Base:\n{knowledge_base}"

print(system_prompt)

You are Ava, the academic advisor assistant for Cascade State University's School of Computer & Data Sciences. You help students with degree requirements, course prerequisites, registration policies, and academic standing, using ONLY the advising knowledge base provided below. Be encouraging, clear, and concise. Ask a clarifying question when the student's request is ambiguous (e.g. which degree program they are in). If a question falls outside the knowledge base (financial aid, medical or personal leave, disciplinary matters, visa/international status, or anything not covered below), do not guess — politely refer the student to the Advising Office at advising@cascadestate.edu.

## Advising Knowledge Base:
Institution: Cascade State University — School of Computer & Data Sciences

DEGREE PROGRAMS
- BS in Computer Science (120 credits): Core sequence is CS101 Intro to Programming -> CS201
  Data Structures -> CS301 Algorithms -> CS310 Systems -> CS401 Capstone. Requires one AI/ML
  elec

## Input Validation

`validate_input` is a single, reusable check shared by the chat function below (and reusable by
any future Gradio callback in this notebook)

In [5]:
MAX_MESSAGE_LENGTH = 1000


def validate_input(message: str) -> str | None:
    """Return a user-facing error string if the message is invalid, else None."""
    message = (message or "").strip()
    if not message:
        return "⚠️ Please type a question before sending."
    if len(message) > MAX_MESSAGE_LENGTH:
        return f"⚠️ Please limit your question to {MAX_MESSAGE_LENGTH} characters."
    return None

In [7]:
# Define the chat function used by Gradio.
def chat(message, history):
    error = validate_input(message)
    if error:
        return error

    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    try:
        response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ Sorry, something went wrong while reaching the advisor assistant: {e}"

## UI Design

The interface is a `gr.ChatInterface`:
- A **title and description** so a first-time visitor understands what Ava can help with
- **Example prompts** so students can start with one click instead of a blank box
- The built-in **Soft theme** for a more polished look, with no extra styling code required

In [ ]:
gr.ChatInterface(
    chat,
    title="🎓 Ava — Academic Advisor Assistant",
    description=(
        "Ask about degree requirements, course prerequisites, registration policies, or "
        "academic standing at Cascade State University's School of Computer & Data Sciences."
    ),
    examples=[
        "What courses do I need for the BS in Computer Science?",
        "What's the prerequisite for Data Structures?",
        "What happens if my GPA drops below 2.0?",
        "How many credits can I take per semester?",
    ],
).launch(share=True, inbrowser=True, inline=False)

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


## Running This Notebook

1. Copy `.env.example` to `.env` and set your `OPENAI_API_KEY`.
2. Select the **Python venv** kernel.
3. Run all cells. The last cell prints a local URL and a public `https://*.gradio.live`